# Gold Layer: `fact_cash_transactions` -- Fact Table

| Property | Value |
|:---------|:------|
| **Target Table** | `gold.fact_cash_transactions` |
| **Expected Rows** | 1,204,943 |
| **Grain** | One row per cash transaction event |
| **MERGE Key** | `(SK_AccountID, TransactionDatetime, Amount)` |
| **Source** | `silver.cash_transactions` |

### Schema
| Column | Type | Description |
|:-------|:-----|:------------|
| `SK_AccountID` | BIGINT | FK to `dim_account` (direct cast from CT_CA_ID) |
| `TransactionDatetime` | TIMESTAMP | Raw transaction timestamp (no SK_DateID/SK_TimeID split) |
| `Amount` | DECIMAL(12,2) | Transaction amount (+deposit, -withdrawal) |
| `Description` | STRING | Transaction description |
| `_batch` | STRING | Batch identifier |

### Design Notes
* **No temporal join** -- SK_AccountID is a direct cast from CT_CA_ID
* **No SK_DateID/SK_TimeID** -- consumers join dim_date/dim_time at query time
* **No dimension dependencies** -- can run in parallel with dim_account
* **Idempotent** -- MERGE on composite key prevents duplicates on rerun

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import (
    current_timestamp, col, count, sum as _sum,
    max as _max, min as _min, lit
)
from pyspark.sql.types import LongType, DecimalType, TimestampType, StringType
from delta.tables import DeltaTable

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONFIGURATION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CATALOG       = "charles_schwab_retailbrokerage_dev_team_lemma"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

SOURCE_TABLE  = f"{CATALOG}.{SILVER_SCHEMA}.cash_transactions"
TARGET_TABLE  = f"{CATALOG}.{GOLD_SCHEMA}.fact_cash_transactions"
EXPECTED_ROWS = 1204943

spark.sql(f"USE CATALOG {CATALOG}")

print("\n" + "="*70)
print("  GOLD.FACT_CASH_TRANSACTIONS -- Pipeline Configuration")
print("="*70)
print(f"  Catalog       : {CATALOG}")
print(f"  Source Table  : {SOURCE_TABLE}")
print(f"  Target Table  : {TARGET_TABLE}")
print(f"  Expected Rows : {EXPECTED_ROWS:,}")
print(f"  MERGE Key     : (SK_AccountID, TransactionDatetime, Amount)")
print(f"  SK_AccountID  : Direct cast from CT_CA_ID (no temporal join)")
print("="*70)

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 1: Read Source -- silver.cash_transactions
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Silver table schema:
#    CT_CA_ID   BIGINT       -> SK_AccountID (direct cast)
#    CT_DTS     TIMESTAMP    -> TransactionDatetime
#    CT_AMT     DECIMAL(12,2)-> Amount
#    CT_NAME    STRING       -> Description
#    _batch     STRING       -> _batch (carried)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 1: Read Source -- silver.cash_transactions")
print("="*70)

silver_df = spark.table(SOURCE_TABLE)
source_count = silver_df.count()

print(f"\n  Source rows: {source_count:,}")
print(f"  Source schema:")
for name, dtype in silver_df.dtypes:
    print(f"    {name:15s} {dtype}")

print(f"\n  Batch distribution:")
silver_df.groupBy("_batch").count().orderBy("_batch").show()

print("  Sample source data (first 10):")
display(silver_df.orderBy("CT_CA_ID", "CT_DTS").limit(10))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 2: Transform to Gold Schema
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Transformation is minimal for this fact table:
#    CT_CA_ID  -> SK_AccountID (direct cast, no temporal join)
#    CT_DTS    -> TransactionDatetime (raw timestamp preserved)
#    CT_AMT    -> Amount (already DECIMAL(12,2) from silver)
#    CT_NAME   -> Description (rename)
#    _batch    -> _batch (carried)
#
#  Per MD: "SK_AccountID is a direct cast from CT_CA_ID
#  (not resolved via temporal join to dim_account)"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 2: Transform to Gold Schema")
print("="*70)

fact_cash_txn_df = silver_df.select(
    col("CT_CA_ID").cast(LongType()).alias("SK_AccountID"),
    col("CT_DTS").alias("TransactionDatetime"),
    col("CT_AMT").alias("Amount"),
    col("CT_NAME").alias("Description"),
    col("_batch")
)

final_count = fact_cash_txn_df.count()
status = "PASS" if final_count == EXPECTED_ROWS else "MISMATCH"

print(f"\n  Transformation applied:")
print(f"    CT_CA_ID  -> SK_AccountID (BIGINT, direct cast)")
print(f"    CT_DTS    -> TransactionDatetime (TIMESTAMP, raw)")
print(f"    CT_AMT    -> Amount (DECIMAL(12,2))")
print(f"    CT_NAME   -> Description (STRING)")
print(f"    _batch    -> _batch (STRING, carried)")

print(f"\n  +{'─'*54}+")
print(f"  |  Final row count : {final_count:>10,}                   |")
print(f"  |  Expected        : {EXPECTED_ROWS:>10,}                   |")
print(f"  |  Status          : {status:>10}                   |")
print(f"  +{'─'*54}+")

print("\n  Schema:")
for name, dtype in fact_cash_txn_df.dtypes:
    print(f"    {name:25s} {dtype}")

print("\n  Sample transformed data (first 10):")
display(fact_cash_txn_df.orderBy("SK_AccountID", "TransactionDatetime").limit(10))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 3: Write to Gold (MERGE for idempotency)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  MERGE ON: (SK_AccountID, TransactionDatetime, Amount)
#  Composite key = natural grain of one transaction event.
#  First run: CREATE via overwrite
#  Subsequent runs: MERGE for idempotent reprocessing
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 3: Write to Gold (Idempotent)")
print("="*70)

if spark.catalog.tableExists(TARGET_TABLE):
    print(f"\n  Mode: MERGE (table exists)")
    print(f"  Key : (SK_AccountID, TransactionDatetime, Amount)")
    
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)
    delta_target.alias("tgt").merge(
        fact_cash_txn_df.alias("src"),
        """tgt.SK_AccountID = src.SK_AccountID 
           AND tgt.TransactionDatetime = src.TransactionDatetime 
           AND tgt.Amount = src.Amount"""
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    
    print("  MERGE complete.")
else:
    print(f"\n  Mode: CREATE (first run)")
    print(f"  Table: {TARGET_TABLE}")
    
    fact_cash_txn_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TARGET_TABLE)
    
    print("  CREATE complete.")

target_count = spark.table(TARGET_TABLE).count()
status = "PASS" if target_count == EXPECTED_ROWS else "FAIL"
print(f"\n  Target count: {target_count:,} (expected: {EXPECTED_ROWS:,}) [{status}]")

print("\n  Delta Table Version History:")
display(spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 5"))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 4: Data Quality Validation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 4: Data Quality Validation")
print("="*70)

gold_df = spark.table(TARGET_TABLE)
results = []

# DQ-1: Row Count
row_count = gold_df.count()
p = row_count == EXPECTED_ROWS
results.append(("DQ-1", "Row Count", f"{row_count:,} vs {EXPECTED_ROWS:,}", "PASS" if p else "FAIL"))

# DQ-2: SK_AccountID Not Null
sk_nulls = gold_df.filter(col("SK_AccountID").isNull()).count()
p = sk_nulls == 0
results.append(("DQ-2", "SK_AccountID Not Null", f"{sk_nulls} nulls", "PASS" if p else "FAIL"))

# DQ-3: TransactionDatetime Not Null
ts_nulls = gold_df.filter(col("TransactionDatetime").isNull()).count()
p = ts_nulls == 0
results.append(("DQ-3", "TransactionDatetime Not Null", f"{ts_nulls} nulls", "PASS" if p else "FAIL"))

# DQ-4: Amount Not Null
amt_nulls = gold_df.filter(col("Amount").isNull()).count()
p = amt_nulls == 0
results.append(("DQ-4", "Amount Not Null", f"{amt_nulls} nulls", "PASS" if p else "FAIL"))

# DQ-5: Composite Key Uniqueness
from pyspark.sql.functions import concat_ws
dup_count = (
    gold_df.groupBy("SK_AccountID", "TransactionDatetime", "Amount")
    .count()
    .filter(col("count") > 1)
    .count()
)
p = dup_count == 0
results.append(("DQ-5", "Composite Key Unique", f"{dup_count} duplicates", "PASS" if p else "WARN"))

# DQ-6: Amount range check (no extreme outliers)
amt_stats = gold_df.select(
    _min("Amount").alias("min_amt"),
    _max("Amount").alias("max_amt")
).collect()[0]
results.append(("DQ-6", "Amount Range", f"min={amt_stats['min_amt']}, max={amt_stats['max_amt']}", "INFO"))

# DQ-7: Date range
date_stats = gold_df.select(
    _min("TransactionDatetime").alias("min_dt"),
    _max("TransactionDatetime").alias("max_dt")
).collect()[0]
results.append(("DQ-7", "Date Range", f"{str(date_stats['min_dt'])[:10]} to {str(date_stats['max_dt'])[:10]}", "INFO"))

# Print results
print("\n  {:<6} {:<30} {:<35} {}".format("Check", "Description", "Result", "Status"))
print("  " + "-"*80)
for check_id, desc, result, status in results:
    print(f"  {check_id:<6} {desc:<30} {result:<35} {status}")

print("\n  Table Detail:")
display(spark.sql(f"DESCRIBE DETAIL {TARGET_TABLE}"))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 5: Operations Logging
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 5: Operations Logging")
print("="*70)

try:
    run_id = spark.table(SOURCE_TABLE).select("_run_id").first()[0]
    
    log_pipeline_recon(
        spark=spark,
        run_id=run_id,
        batch_id="ALL",
        domain="ACCOUNT",
        table_name="fact_cash_transactions",
        source_layer="silver",
        target_layer="gold",
        source_count=source_count,
        target_count=target_count
    )
    
    log_audit_event(
        spark=spark,
        run_id=run_id,
        batch="ALL",
        layer="gold",
        table_name="fact_cash_transactions",
        operation="MERGE" if spark.catalog.tableExists(TARGET_TABLE) else "CREATE",
        rows_affected=target_count
    )
    
    print(f"\n  log_pipeline_recon: source={source_count:,} -> target={target_count:,}")
    print(f"  log_audit_event: rows={target_count:,}")
except Exception as e:
    print(f"  [WARN] Operations logging failed: {e}")
    print(f"  (Non-blocking -- gold table was written successfully)")

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  VERIFICATION: Final State Summary
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  VERIFICATION: gold.fact_cash_transactions Final State")
print("="*70)

final_df = spark.table(TARGET_TABLE)

print(f"\n  Total rows: {final_df.count():,} (expected: {EXPECTED_ROWS:,})")

print("\n  Batch Distribution:")
final_df.groupBy("_batch").count().orderBy("_batch").show()

print("  Amount Statistics:")
display(
    final_df.select(
        count("*").alias("total_rows"),
        _sum("Amount").alias("total_amount"),
        _min("Amount").alias("min_amount"),
        _max("Amount").alias("max_amount"),
        _sum(col("Amount").cast("int")).alias("sum_int_approx")
    )
)

print("  Top 10 accounts by transaction count:")
display(
    final_df.groupBy("SK_AccountID")
    .agg(
        count("*").alias("txn_count"),
        _sum("Amount").alias("net_amount")
    )
    .orderBy(col("txn_count").desc())
    .limit(10)
)

print("  Sample transactions (first 15):")
display(final_df.orderBy("SK_AccountID", "TransactionDatetime").limit(15))